In [12]:
import os
import math
import random
import numpy as np
import pandas as pd
import fiona
import logging
import fiona

import geopandas as gpd
from shapely.geometry import Point, Polygon, MultiPoint, MultiPolygon, LineString
from shapely.ops import unary_union, voronoi_diagram, nearest_points
from shapely import affinity

In [ ]:
RP = ['100']
RCP = ['baseline2010', '262050', '262100', '452030', '452050', '452070', '452100', '852030', '852050', '852070', '852100']


#------------------ FILE PATHS -----------------------
networks_csv = "data/network_layers.csv"
data_path = "/Users/adam/PROJECTS/JSRAT/JSRAT-GITS/jamaica-infrastructure/processed_data/"
networks = pd.read_csv(networks_csv)

flood_areas = "outputs/Jamaica_coastal_protection_areas.gpkg"
# flood_layer = "flood_protection_area_rcp_baseline2010_rp100"

# coastal_buffer = "outputs/coastal_protection_processing_layers.gpkg"
# coastal_buffer_layer = "jamaica_convex"
#------------------------------------------------------

In [6]:
def join_flood_areas(flood_polygons):
    # Merge all polygons into a single polygon
    merged_polygon = unary_union(flood_polygons.geometry)
    return merged_polygon

def find_intersecting_assets(flood_polygon, layer):
    # Ensure both layer and flood_polygon are in the same CRS
    layer = layer.to_crs(flood_polygon.crs)
    
    # Perform the spatial join where the geometries intersect
    intersecting_assets = gpd.sjoin(layer, flood_polygon, how="inner", predicate="intersects")
    
    return intersecting_assets

all_flood_areas = []

for rcp in RCP:
    for rp in RP:
        flood_polygon_layer = f"flood_protection_area_rcp_{rcp}_rp{rp}"
        flood_polygons = gpd.read_file(flood_areas, layer=flood_polygon_layer)
        merged_flood_polygon = join_flood_areas(flood_polygons)
        all_flood_areas.append(merged_flood_polygon)

# Create a union of all merged flood areas
final_flood_area = unary_union(all_flood_areas)

# Save the final merged flood polygon
union_flood_area_gdf = gpd.GeoDataFrame(geometry=[final_flood_area], crs=flood_polygons.crs)
union_flood_area_gdf.to_file("outputs/full_coastal_area.gpkg", driver="GPKG")




In [14]:
#Change to alter what netwirk layers are processed
network_lst = ["transport_roads_edges"]

In [ ]:
# Assuming union_flood_area_gdf is already defined
flood_polygon = union_flood_area_gdf

# Take just the first (or main) geometry from the flood union
flood = flood_polygon.loc[flood_polygon.index[0]]
flood_polygon = gpd.GeoDataFrame([flood], geometry='geometry', crs=flood_polygon.crs)

for index, n in networks.iterrows():
    fname = os.path.join(data_path, n['path'])
    id_col = n['asset_id_column']
    layer_type = n['gpkg_layer']
    ref = n['ref']

    if ref in network_lst:
        continue

    # Load the asset layer
    assets = gpd.read_file(fname, layer=layer_type)
    if assets.empty:
        continue

    # Find intersecting assets
    asset_intersections = find_intersecting_assets(flood_polygon, assets)
    
    # Keep only the ID column and drop duplicates
    filtered_ids = asset_intersections[[id_col]].drop_duplicates()

    # Save to Parquet
    parquet_path = os.path.join("outputs/asset_parquets", f"{ref}_coastal_filtered.parquet")
    filtered_ids.to_parquet(parquet_path, index=False)
    
    print("Completed for", ref, "- saved to", parquet_path)


Completed for transport_rail_nodes - saved to outputs/transport_rail_nodes_coastal_filtered.parquet
Completed for transport_rail_edges - saved to outputs/transport_rail_edges_coastal_filtered.parquet
Completed for transport_air - saved to outputs/transport_air_coastal_filtered.parquet
Completed for transport_port - saved to outputs/transport_port_coastal_filtered.parquet
Completed for water_irrigation_nodes - saved to outputs/water_irrigation_nodes_coastal_filtered.parquet
Completed for water_irrigation_edges - saved to outputs/water_irrigation_edges_coastal_filtered.parquet
Completed for water_potable_nodes - saved to outputs/water_potable_nodes_coastal_filtered.parquet
Completed for water_wastewater_nodes - saved to outputs/water_wastewater_nodes_coastal_filtered.parquet
Completed for water_pipelines - saved to outputs/water_pipelines_coastal_filtered.parquet
Completed for transport_roads_nodes - saved to outputs/transport_roads_nodes_coastal_filtered.parquet
Completed for energy_nod

In [9]:
def Assign_flood_area_to_asset(asset_network, RCP, RP, path, id_label):
    def find_flood_area_asset_intersection(flood_polygons, asset):
        asset_geom = asset.geometry
        intersecting_floods = flood_polygons[flood_polygons.geometry.intersects(asset_geom)]
        
        if not intersecting_floods.empty:
            max_flood_polygon = intersecting_floods.loc[intersecting_floods['max_flood_height'].idxmax()]
            flood_polygon_id = max_flood_polygon['id']
            max_flood_height = max_flood_polygon['max_flood_height']
        else:
            flood_polygon_id = None
            max_flood_height = None

        return flood_polygon_id, max_flood_height

    df = pd.read_parquet(path)
    
    all_layers = fiona.listlayers(flood_areas)
    flood_layers = {
        layer: gpd.read_file(flood_areas, layer=layer)
        for layer in all_layers if layer.startswith("flood_protection_area_rcp_")
    }

    previous_layer_name = None

    for i, row in df.iterrows():
        asset_id = row.iloc[0]  

        print (f"Current asset: {asset_id}")
        for rcp in RCP:
            for rp in RP:
                layer_name = f"flood_protection_area_rcp_{rcp}_rp{rp}"
        
                # if layer_name != previous_layer_name:
                    # print(f"Current Layer: {layer_name}")
                
                previous_layer_name = layer_name
                
                flood_polygons = flood_layers.get(layer_name)
                asset = asset_network[asset_network[id_label] == asset_id].iloc[0]
                flood_id, flood_height = find_flood_area_asset_intersection(flood_polygons, asset)
                flood_id_col, flood_height_col = f"flood_id_rcp_{rcp}_rp_{rp}", f"flood_height_rcp_{rcp}_rp_{rp}"

                df.at[i, flood_id_col] = flood_id
                df.at[i, flood_height_col] = flood_height

    return df

In [15]:
RCP = ['baseline2010', '262100', '452030', '452050', '452070', '452100', '852030', '852050', '852070', '852100']
# RCP = ['262100']
RP = ['100']
output_path = 'outputs/asset_parquets/'

for index, n in networks.iterrows():
    fname = os.path.join(data_path, n['path'])
    id_col = n['asset_id_column']
    layer_type = n['gpkg_layer']
    ref = n['ref']
    path = f'{output_path}{ref}_coastal_filtered.parquet'

    if ref not in network_lst:
        continue

    # Load the asset layer
    assets = gpd.read_file(fname, layer=layer_type)
    if assets.empty:
        continue

    updated_output = Assign_flood_area_to_asset(assets, RCP, RP, path, id_col)
    updated_output.to_parquet(path, index=False) 


Current asset: roade_1
Current asset: roade_2
Current asset: roade_3
Current asset: roade_4
Current asset: roade_5
Current asset: roade_12
Current asset: roade_13
Current asset: roade_14
Current asset: roade_15
Current asset: roade_16
Current asset: roade_17
Current asset: roade_18
Current asset: roade_45
Current asset: roade_46
Current asset: roade_47
Current asset: roade_51
Current asset: roade_52
Current asset: roade_53
Current asset: roade_54
Current asset: roade_55
Current asset: roade_56
Current asset: roade_110
Current asset: roade_114
Current asset: roade_115
Current asset: roade_116
Current asset: roade_117
Current asset: roade_118
Current asset: roade_119
Current asset: roade_120
Current asset: roade_121
Current asset: roade_122
Current asset: roade_123
Current asset: roade_124
Current asset: roade_125
Current asset: roade_126
Current asset: roade_127
Current asset: roade_128
Current asset: roade_129
Current asset: roade_130
Current asset: roade_141
Current asset: roade_142
C